# Quality Checks: Bronze tables

### 

## Table: bronze.crm.customer_info

In [ ]:
%%sql
-- check for NULLS or duplicates in primary key column (cst_id)
SELECT cst_id, COUNT(*) AS id_count FROM sales_lakehouse.dbo.bronze_crm_customer_info
GROUP BY cst_id
HAVING COUNT(*) > 1 OR cst_id IS NULL;

-- Observation: duplicate and NULLS found in cst_id column (primary key)

In [ ]:
-- check for unwanted spaces in string columns
-- check for unwanted spaces in  cst_firstname
SELECT 
cst_firstname
FROM sales_lakehouse.dbo.bronze_crm_customer_info
WHERE cst_firstname != TRIM(cst_firstname);

-- Observation: the query returns the firstnames with leading/trailing spaces in cst_firstname column

In [ ]:
-- check for unwanted spaces in cst_lastname
SELECT 
cst_lastname
FROM sales_lakehouse.dbo.bronze_crm_customer_info
WHERE cst_lastname != TRIM(cst_lastname);

-- Observation: the query returns the lastnames with leading/trailing spaces in cst_lastname column

In [ ]:
-- Qulity issues in low cardinality columns (Data Standardization & Consistency)
-- cst_marital_status
SELECT DISTINCT cst_marital_status
FROM sales_lakehouse.dbo.bronze_crm_customer_info;

-- observation: found NULL

In [ ]:
-- cst_gndr column
SELECT DISTINCT cst_gndr
FROM sales_lakehouse.dbo.bronze_crm_customer_info;

-- observation: found NULL

In [ ]:
-- verifying schema 
DESCRIBE TABLE sales_lakehouse.dbo.bronze_crm_customer_info;

## Table: bronze.crm.product_info

In [ ]:
-- check for NULLS or duplicates in primary key column (prd_id)
SELECT prd_id, COUNT(*) AS id_count FROM sales_lakehouse.dbo.bronze_crm_product_info
GROUP BY prd_id
HAVING COUNT(*) > 1 OR prd_id IS NULL;

-- -- observation: No duplicates or NULLS found

In [ ]:
-- check whether all the category ids exist in product category table after creating the cat_id in crm_product_info
SELECT 
    prd_id,
    prd_key,
    REPLACE(SUBSTRING(prd_key, 1, 5), "-", "_") AS cat_id, -- this one is used to merge with erp_product_category
    prd_nm,
    prd_cost,
    prd_line,
    prd_start_dt,
    prd_end_dt
FROM sales_lakehouse.dbo.bronze_crm_product_info
WHERE REPLACE(SUBSTRING(prd_key, 1, 5), "-", "_") NOT IN
(SELECT DISTINCT ID FROM sales_lakehouse.dbo.bronze_erp_product_category);

-- Observation: CO_PE cat_id is there in bronze_crm_product_info but NOT in erp_product_category, which is fine and possible

In [ ]:
-- check whether all the product ids exist in bronze_crm_sales_details table after creating the prd_id in crm_product_info
SELECT 
    prd_id,
    prd_key,
    REPLACE(SUBSTRING(prd_key, 1, 5), "-", "_") AS cat_id,      
    SUBSTRING(prd_key, 7, LEN(prd_key)) AS prd_key,         -- this key will be used to join with crm_sales_details
    prd_nm,
    prd_cost,
    prd_line,
    prd_start_dt,
    prd_end_dt
FROM sales_lakehouse.dbo.bronze_crm_product_info
WHERE SUBSTRING(prd_key, 7, LEN(prd_key)) NOT IN
(SELECT sls_prd_key FROM sales_lakehouse.dbo.bronze_crm_sales_details);

-- Observation: Few prd_key found. They may not have sales in crm_sales_details table, which is fine

In [ ]:
-- check for unwanted spaces in string columns

SELECT
prd_nm
FROM sales_lakehouse.dbo.bronze_crm_product_info
WHERE prd_nm != TRIM(prd_nm);

-- Observation: NO leading and trailing spaces

In [ ]:
-- check for NULLS or Negative Numbers
SELECT
prd_cost
FROM sales_lakehouse.dbo.bronze_crm_product_info
WHERE prd_cost < 0 OR prd_cost IS NULL;

-- Observation: NULLs found

In [ ]:
-- check issues in low cardinality columns
-- Data Standardization & Consistency
SELECT
DISTINCT prd_line
FROM sales_lakehouse.dbo.bronze_crm_product_info;

In [ ]:
-- check for invalid date orders
-- End date must not be smaller than the start date
SELECT
*
FROM sales_lakehouse.dbo.bronze_crm_product_info
WHERE prd_end_dt < prd_start_dt;

-- Observation: Few End dates are smaller than the start date.
-- Solution: End Date = Start Date of the Next Record - 1 (there shouldn't be any overlapping dates)

In [ ]:
-- Checking few records where prd_end_dt < prd_start_dt
SELECT
*
FROM sales_lakehouse.dbo.bronze_crm_product_info
WHERE prd_key IN ('AC-HE-HL-U509-R', 'AC-HE-HL-U509');

## Table: bronze.crm_sales_details

In [ ]:
-- check for unwanted spaces in string columns

SELECT
sls_ord_num 
FROM sales_lakehouse.dbo.bronze_crm_sales_details
WHERE sls_ord_num != TRIM(sls_ord_num);

-- observation: No spaces

In [ ]:
-- check if all key values from sales table are there in product table

SELECT
* 
FROM sales_lakehouse.dbo.bronze_crm_sales_details
WHERE sls_prd_key NOT IN 
(SELECT prd_key FROM sales_lakehouse.dbo.silver_crm_product_info);

-- observation: no result. all keys from sales table are there in product table

In [ ]:
-- check if all key values from sales table are there in customer table

SELECT
* 
FROM sales_lakehouse.dbo.bronze_crm_sales_details
WHERE sls_cust_id NOT IN 
(SELECT cst_id FROM sales_lakehouse.dbo.silver_crm_customer_info);

-- observation: no result. all keys from sales table are there in customer table

#### check for invalid dates
* Negative numbes or zeros can't be cast to a date
* length of the date must be 8 here
* check for outliers by validating the boundaries of the date range

###### `sls_order_dt`

In [ ]:
SELECT
sls_order_dt             -- 
FROM sales_lakehouse.dbo.bronze_crm_sales_details
WHERE sls_order_dt <= 0
OR LEN(sls_order_dt) != 8
OR sls_order_dt > 20500101
OR sls_order_dt < 19000101;

-- observation: 
-- there are zeros in sls_order_dt -- this is to be converted as NULL
-- length is != 8 (these are not dates)
-- random date range (19000101 to 20500101) - no date is outside this range.

###### `sls_ship_dt`

In [ ]:
SELECT
sls_ship_dt             -- 
FROM sales_lakehouse.dbo.bronze_crm_sales_details
WHERE sls_ship_dt <= 0
OR LEN(sls_ship_dt) != 8
OR sls_ship_dt > 20500101
OR sls_ship_dt < 19000101;

-- observation: no result (no issues found) 

###### `sls_due_dt`

In [ ]:
SELECT
sls_due_dt             -- 
FROM sales_lakehouse.dbo.bronze_crm_sales_details
WHERE sls_due_dt <= 0
OR LEN(sls_due_dt) != 8
OR sls_due_dt > 20500101
OR sls_due_dt < 19000101;

-- observation: no result (no issues found) 

In [ ]:
-- check for invalid date orders
-- Order Date must always be earlier than the Shipping Date or Due Date
SELECT
*             -- 
FROM sales_lakehouse.dbo.bronze_crm_sales_details
WHERE sls_order_dt > sls_ship_dt OR sls_order_dt > sls_due_dt;

-- observation: two sls_prd_key (TI-M823 and TT-M928). 
-- The sls_order_dt (32154 and 5489) will not considered as dates since the length is !=8 (will be replaced with NULL)

In [ ]:
-- Check Data Consistency between Sales, Quantity and Price
--Business Rule:
--Sales = Quantity * Price
--Negative, zeros and NULLs are NOT Allowed in neither in Sales, Quantity and Price

SELECT
sls_sales,
sls_quantity,
sls_price
FROM sales_lakehouse.dbo.bronze_crm_sales_details
WHERE sls_sales != sls_quantity * sls_price
OR sls_sales IS NULL OR sls_quantity IS NULL OR sls_price IS NULL
OR sls_sales <= 0 OR sls_quantity <= 0 OR sls_price <= 0
ORDER BY sls_sales, sls_quantity, sls_price;

-- observation: we have NULLS, zeros and negative numbers.
-- We need to define the rule to fix these issues
-- 1. If Sales is negative, zero or null, derive it using Quantity and Price
-- 2. If Price is zero or null, calculate it using Sales and Quanity
-- 3. If Price is negative, convert it to a positive value


In [ ]:
-- check after applying the rules
SELECT
sls_sales as sls_sales_old,
sls_quantity as sls_quantity_old,
sls_price as sls_price_old,

CASE WHEN sls_sales <= 0 OR sls_sales IS NULL OR sls_sales != sls_quantity * ABS(sls_price)
     THEN CAST(sls_quantity * ABS(sls_price) AS INT)
     ELSE sls_sales
END AS sls_sales,

CASE WHEN sls_price <= 0 OR sls_price IS NULL
     THEN CAST(sls_sales / NULLIF(sls_quantity,0) AS INT)        -- handling divide by 0 error
     ELSE sls_price
END AS sls_price

FROM sales_lakehouse.dbo.bronze_crm_sales_details
WHERE sls_sales != sls_quantity * sls_price
OR sls_sales IS NULL OR sls_quantity IS NULL OR sls_price IS NULL
OR sls_sales <= 0 OR sls_quantity <= 0 OR sls_price <= 0
ORDER BY sls_sales, sls_quantity, sls_price;

## Table: bronze_erp_customers

In [ ]:
-- checking duplicates in bronze_erp_customers
SELECT
cid,
COUNT(*) as cid_count
FROM sales_lakehouse.dbo.bronze_erp_customers
GROUP BY cid
HAVING COUNT(*) > 1;

-- observation: no duplicates

In [ ]:
-- check if all cid is there in silver_crm_customer_info
SELECT
cid
FROM sales_lakehouse.dbo.bronze_erp_customers 
WHERE cid NOT IN
(SELECT DISTINCT cst_key FROM sales_lakehouse.dbo.silver_crm_customer_info);

-- observation: not mathcing cid found. It is because, these cids starts with 'NAS'
-- 'NAS' to be removed from cid to match the cst_key from silver_crm_customer_info

In [ ]:
-- check outliers in bdate column
SELECT
bdate
FROM sales_lakehouse.dbo.bronze_erp_customers
WHERE bdate < '1920-01-01' OR bdate > CURRENT_TIMESTAMP();

-- observation: 
-- there are birth dates which are less than the random 1920-01-01 (customers more than 100 years old)
-- birth dates which is in future (not possible)

In [ ]:
-- check low cardinality columns
SELECT
DISTINCT gen
FROM sales_lakehouse.dbo.bronze_erp_customers;

-- observation: gender column don't have standard values

## Table: bronze_erp_location

In [ ]:
-- check if all keys are present in silver_crm_

SELECT 
cid,
COUNT(*) as cid_count
FROM sales_lakehouse.dbo.bronze_erp_location
GROUP BY cid
HAVING COUNT(*) > 1;

-- observation: no duplicates

In [ ]:
SELECT 
cid
FROM sales_lakehouse.dbo.bronze_erp_location
WHERE cid NOT IN
(SELECT cst_key FROM sales_lakehouse.dbo.silver_crm_customer_info);

-- observation: cid has '-' because of which there is a mismatch.
-- Hyphen '-' to be removed

In [ ]:
-- check if the transformation will fix the issue
SELECT 
REPLACE(cid, '-', '') AS cid
FROM sales_lakehouse.dbo.bronze_erp_location
WHERE REPLACE(cid, '-', '') NOT IN
(SELECT cst_key FROM sales_lakehouse.dbo.silver_crm_customer_info);

-- observation: no result. Hence, transformation will fix the issue

In [ ]:
-- check low cardinality columns
SELECT 
DISTINCT cntry
FROM sales_lakehouse.dbo.bronze_erp_location;

-- observation: country column values are not standardized

## Table: bronze_erp_product_category

In [ ]:
-- check if all id is matching with the silver table (which will be used to join later)
SELECT
id
FROM sales_lakehouse.dbo.bronze_erp_product_category
WHERE id NOT IN
(SELECT cat_id FROM sales_lakehouse.dbo.silver_crm_product_info);

-- observation: this is possible and fine. (we found this bronze_crm_product_info quality check)

In [ ]:
-- checking unwanted spaces
SELECT
cat
FROM sales_lakehouse.dbo.bronze_erp_product_category
WHERE cat != TRIM(cat);

-- observation: no spaces

In [ ]:
-- checking unwanted spaces
SELECT
subcat
FROM sales_lakehouse.dbo.bronze_erp_product_category
WHERE subcat != TRIM(subcat);

-- observation: no spaces

In [ ]:
-- checking unwanted spaces
SELECT
maintenance
FROM sales_lakehouse.dbo.bronze_erp_product_category
WHERE maintenance != TRIM(maintenance);

-- observation: no spaces

In [ ]:
-- checking low cardinality columns
SELECT
DISTINCT cat
FROM sales_lakehouse.dbo.bronze_erp_product_category;

-- observation: Looks good. Nothing to fix

In [ ]:
-- checking low cardinality columns
SELECT
DISTINCT subcat
FROM sales_lakehouse.dbo.bronze_erp_product_category;

-- observation: Looks good. Nothing to fix

In [ ]:
-- checking low cardinality columns
SELECT
DISTINCT maintenance
FROM sales_lakehouse.dbo.bronze_erp_product_category;

-- observation: Looks good. Nothing to fix